# RayNet v2 Training on Google Colab A100

**Hardware target:** NVIDIA A100 (80 GB VRAM) + ~140 GB system RAM

**Features:**
- WebDataset streaming from Hugging Face Hub
- Mixed precision (fp16) training with GradScaler
- TF32 for matmul acceleration
- torch.compile for kernel fusion
- Gradient accumulation for effective batch size of 4096
- Multi-view consistency loss (Phases 2-3)

**Runtime:** Change runtime to **A100 GPU** before running.

## 1. Setup & Install

In [ ]:
# Verify GPU
!nvidia-smi

import torch
assert torch.cuda.is_available(), "No GPU detected. Change runtime to A100."
gpu_name = torch.cuda.get_device_name(0)
gpu_mem_gb = torch.cuda.get_device_properties(0).total_mem / 1e9
print(f"GPU: {gpu_name} ({gpu_mem_gb:.1f} GB)")
assert gpu_mem_gb > 30, f"Expected A100 (80GB), got {gpu_name} ({gpu_mem_gb:.0f}GB)"

In [ ]:
# Install dependencies
!pip install -q webdataset huggingface_hub timm opencv-python-headless

# Clone the repo (or mount from Drive)
import os
REPO_DIR = '/content/RayNet'
if not os.path.exists(REPO_DIR):
    # Option A: Clone from GitHub
    !git clone https://github.com/YOUR_USERNAME/RayNet.git {REPO_DIR}
    # Option B: Mount Google Drive
    # from google.colab import drive
    # drive.mount('/content/drive')
    # REPO_DIR = '/content/drive/MyDrive/RayNet'

os.chdir(REPO_DIR)
!pip install -q -r requirements.txt

import sys
sys.path.insert(0, REPO_DIR)

In [ ]:
# Hugging Face authentication (for private datasets)
from huggingface_hub import notebook_login
notebook_login()

## 2. A100-Optimized Configuration

In [ ]:
# ====================================================================
# A100 80GB + 140GB RAM Configuration
# ====================================================================

CONFIG = {
    # --- Model ---
    'backbone': 'repnext_m3',
    'weight_path': None,          # set to pretrained backbone path if available
    'n_landmarks': 14,

    # --- Data ---
    'eye': 'L',
    'samples_per_subject': None,  # None = ALL frames (leverage 140GB RAM)
    'img_size': 224,

    # --- A100 Hardware ---
    'batch_size': 2048,           # fits in 80GB with fp16
    'mv_groups': 16,              # 144 samples per multi-view batch
    'num_workers': 8,             # Colab has 12 CPU cores
    'prefetch_factor': 4,
    'persistent_workers': True,
    'pin_memory': True,

    # --- Mixed Precision ---
    'amp': True,
    'amp_dtype': torch.float16,   # A100 has excellent fp16 throughput

    # --- Gradient Accumulation ---
    'grad_accum_steps': 2,        # effective batch = 4096

    # --- Compilation ---
    'compile_model': True,        # torch.compile: fuses kernels on A100
    'tf32': True,                 # TF32 for float32 matmul (19x speedup)

    # --- Training ---
    'epochs': 30,
    'output_dir': '/content/results',

    # --- Streaming (set your HF repo) ---
    'streaming': True,
    'hf_repo_id': 'YOUR_USERNAME/gazegene-webdataset',  # <-- EDIT THIS
    'n_train_shards': None,       # auto-detect or set manually
    'n_val_shards': None,
}

# Phase schedule (same as train.py)
PHASE_CONFIG = {
    1: {'epochs': (1, 5),   'lam_lm': 1.0, 'lam_gaze': 0.0, 'lam_reproj': 0.0,  'lam_mask': 0.0,  'lr': 1e-3, 'sigma': 2.0, 'multiview': False},
    2: {'epochs': (6, 15),  'lam_lm': 1.0, 'lam_gaze': 0.3, 'lam_reproj': 0.1,  'lam_mask': 0.05, 'lr': 5e-4, 'sigma': 1.5, 'multiview': True},
    3: {'epochs': (16, 30), 'lam_lm': 0.5, 'lam_gaze': 0.5, 'lam_reproj': 0.2,  'lam_mask': 0.1,  'lr': 1e-4, 'sigma': 1.0, 'multiview': True},
}

print("Configuration loaded.")
print(f"  Effective batch size: {CONFIG['batch_size'] * CONFIG['grad_accum_steps']}")
print(f"  Multi-view groups: {CONFIG['mv_groups']} ({CONFIG['mv_groups'] * 9} samples)")

## 3. Hardware Setup

In [ ]:
import torch
import psutil

# Enable TF32
if CONFIG['tf32']:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.set_float32_matmul_precision('high')
    print("TF32 enabled")

# Report resources
device = torch.device('cuda')
gpu_props = torch.cuda.get_device_properties(0)
ram_gb = psutil.virtual_memory().total / 1e9

print(f"GPU: {gpu_props.name}")
print(f"  VRAM: {gpu_props.total_mem / 1e9:.1f} GB")
print(f"  SM count: {gpu_props.multi_processor_count}")
print(f"  Compute capability: {gpu_props.major}.{gpu_props.minor}")
print(f"System RAM: {ram_gb:.1f} GB")
print(f"CPU cores: {os.cpu_count()}")

## 4. Dataset Loading

Two paths supported:
- **Path A**: Stream from HF Hub (recommended for Colab)
- **Path B**: Local dataset (if uploaded to Colab disk/Drive)

In [ ]:
# ====================================================================
# Path A: Stream from Hugging Face Hub
# ====================================================================

if CONFIG['streaming']:
    from RayNet.webdataset_utils import (
        create_streaming_dataloader,
        create_multiview_streaming_dataloader,
        hf_hub_shard_urls,
    )

    repo_id = CONFIG['hf_repo_id']

    # Build URLs — adjust n_shards to match your dataset
    train_url = hf_hub_shard_urls(repo_id, split='train',
                                   n_shards=CONFIG['n_train_shards'] or 100)
    val_url = hf_hub_shard_urls(repo_id, split='val',
                                 n_shards=CONFIG['n_val_shards'] or 20)

    # Standard loader (Phase 1)
    train_loader_standard = create_streaming_dataloader(
        urls=train_url,
        batch_size=CONFIG['batch_size'],
        num_workers=CONFIG['num_workers'],
        shuffle=True,
    )

    # Multi-view loader (Phases 2-3)
    train_loader_mv = create_multiview_streaming_dataloader(
        urls=train_url,
        mv_groups=CONFIG['mv_groups'],
        num_workers=CONFIG['num_workers'],
        shuffle=True,
    )

    # Validation loader
    val_loader = create_streaming_dataloader(
        urls=val_url,
        batch_size=CONFIG['batch_size'],
        num_workers=CONFIG['num_workers'],
        shuffle=False,
    )

    print(f"Streaming from: {repo_id}")
    print(f"  Train URL: {train_url[:80]}...")

else:
    # ====================================================================
    # Path B: Local dataset
    # ====================================================================
    from RayNet.dataset import create_dataloaders

    DATA_DIR = '/content/gazegene'  # <-- EDIT THIS

    train_loader_standard, val_loader = create_dataloaders(
        base_dir=DATA_DIR,
        train_subjects=list(range(1, 47)),
        val_subjects=list(range(47, 57)),
        batch_size=CONFIG['batch_size'],
        num_workers=CONFIG['num_workers'],
        samples_per_subject=CONFIG['samples_per_subject'],
        eye=CONFIG['eye'],
        ensure_multiview=False,
    )

    train_loader_mv, _ = create_dataloaders(
        base_dir=DATA_DIR,
        train_subjects=list(range(1, 47)),
        val_subjects=list(range(47, 57)),
        batch_size=CONFIG['mv_groups'],
        num_workers=CONFIG['num_workers'],
        samples_per_subject=CONFIG['samples_per_subject'],
        eye=CONFIG['eye'],
        ensure_multiview=True,
    )

    print(f"Local dataset: {DATA_DIR}")
    print(f"  Train samples: {len(train_loader_standard.dataset)}")
    print(f"  Val samples: {len(val_loader.dataset)}")

## 5. Model Creation

In [ ]:
from RayNet.raynet import create_raynet

model = create_raynet(
    backbone_name=CONFIG['backbone'],
    weight_path=CONFIG['weight_path'],
    n_landmarks=CONFIG['n_landmarks'],
)

# torch.compile for A100 kernel fusion
if CONFIG['compile_model']:
    print("Compiling model with torch.compile...")
    model = torch.compile(model)
    print("Done. First forward pass will trigger compilation (may take ~60s).")

# Count parameters
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model: {CONFIG['backbone']} | {n_params/1e6:.1f}M trainable params")

## 6. Training Loop

In [ ]:
import csv
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.cuda.amp import GradScaler, autocast
from datetime import datetime

from RayNet.losses import total_loss
from RayNet.multiview_loss import multiview_consistency_loss


def get_phase_config(epoch):
    for phase, cfg in PHASE_CONFIG.items():
        start, end = cfg['epochs']
        if start <= epoch <= end:
            return phase, cfg
    return 3, PHASE_CONFIG[3]


# --- Setup ---
scaler = GradScaler(enabled=CONFIG['amp'])
amp_dtype = CONFIG['amp_dtype']
grad_accum = CONFIG['grad_accum_steps']

output_dir = os.path.join(CONFIG['output_dir'],
                           f"run_{datetime.now().strftime('%Y%m%d_%H%M%S')}")
os.makedirs(output_dir, exist_ok=True)

# CSV logger
csv_path = os.path.join(output_dir, 'training_log.csv')
csv_file = open(csv_path, 'w', newline='')
csv_writer = csv.writer(csv_file)
csv_writer.writerow([
    'epoch', 'phase', 'lr',
    'train_total', 'train_lm', 'train_ang_deg',
    'train_reproj', 'train_mask',
    'val_total', 'val_lm', 'val_ang_deg', 'val_lm_px',
])

best_val_loss = float('inf')
current_phase = 0
history = {'train_total': [], 'val_total': [], 'train_ang': [], 'val_ang': []}

print(f"Training for {CONFIG['epochs']} epochs")
print(f"Output: {output_dir}")
print(f"AMP: {CONFIG['amp']} | Grad accum: {grad_accum} | "
      f"Effective batch: {CONFIG['batch_size'] * grad_accum}")
print()

In [ ]:
for epoch in range(1, CONFIG['epochs'] + 1):
    phase, cfg = get_phase_config(epoch)
    use_mv = cfg.get('multiview', False) and cfg.get('lam_reproj', 0) > 0

    # --- Phase transition ---
    if phase != current_phase:
        current_phase = phase
        print(f"\n{'='*60}")
        print(f"Phase {phase} | lr={cfg['lr']} | "
              f"lam_lm={cfg['lam_lm']} lam_gaze={cfg['lam_gaze']} | "
              f"multiview={use_mv}")
        if use_mv:
            print(f"  lam_reproj={cfg['lam_reproj']} lam_mask={cfg['lam_mask']}")
        print(f"{'='*60}")

        optimizer = optim.AdamW(
            model.parameters(), lr=cfg['lr'],
            betas=(0.5, 0.95), weight_decay=1e-4)
        start, end = cfg['epochs']
        scheduler = CosineAnnealingLR(optimizer, T_max=end - start + 1)

    # --- Select dataloader ---
    train_loader = train_loader_mv if use_mv else train_loader_standard

    # === TRAIN ===
    model.train()
    train_metrics = {'total': 0, 'lm': 0, 'ang': 0, 'ang_deg': 0, 'reproj': 0, 'mask': 0}
    n_batches = 0
    optimizer.zero_grad()

    for step, batch in enumerate(train_loader):
        images = batch['image'].to(device, non_blocking=True)
        gt_lm = batch['landmark_coords'].to(device, non_blocking=True)
        gt_gaze = batch['optical_axis'].to(device, non_blocking=True)

        with autocast(device_type='cuda', dtype=amp_dtype, enabled=CONFIG['amp']):
            preds = model(images)
            feat_H = preds['landmark_heatmaps'].shape[2]

            loss, comp = total_loss(
                preds['landmark_heatmaps'], preds['landmark_coords'],
                preds['gaze_vector'], gt_lm, gt_gaze,
                feat_H, feat_H,
                lam_lm=cfg['lam_lm'], lam_gaze=cfg['lam_gaze'],
                sigma=cfg['sigma'])

            if use_mv:
                mv_loss, mv_comp = multiview_consistency_loss(
                    preds['landmark_coords'],
                    batch_meta={
                        'K': batch['K'].to(device, non_blocking=True),
                        'R_cam': batch['R_cam'].to(device, non_blocking=True),
                        'T_cam': batch['T_cam'].to(device, non_blocking=True),
                        'M_norm_inv': batch['M_norm_inv'].to(device, non_blocking=True),
                        'eyeball_center_3d': batch['eyeball_center_3d'].to(device, non_blocking=True),
                    },
                    lam_reproj=cfg['lam_reproj'], lam_mask=cfg['lam_mask'],
                    img_size=224, feat_size=feat_H)
                loss = loss + mv_loss
                train_metrics['reproj'] += mv_comp['reproj_loss'].item()
                train_metrics['mask'] += mv_comp['mask_loss'].item()

            loss = loss / grad_accum

        scaler.scale(loss).backward()

        if (step + 1) % grad_accum == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        train_metrics['total'] += comp['total_loss'].item()
        train_metrics['lm'] += comp['landmark_loss'].item()
        train_metrics['ang'] += comp['angular_loss'].item()
        train_metrics['ang_deg'] += comp['angular_loss_deg'].item()
        n_batches += 1

    # Flush remaining gradients
    if n_batches % grad_accum != 0:
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()

    for k in train_metrics:
        train_metrics[k] /= max(n_batches, 1)

    # === VALIDATE ===
    model.eval()
    val_metrics = {'total': 0, 'lm': 0, 'ang_deg': 0, 'lm_px': 0}
    n_val = 0

    with torch.no_grad():
        for batch in val_loader:
            images = batch['image'].to(device, non_blocking=True)
            gt_lm = batch['landmark_coords'].to(device, non_blocking=True)
            gt_gaze = batch['optical_axis'].to(device, non_blocking=True)
            gt_lm_px = batch['landmark_coords_px'].to(device, non_blocking=True)

            with autocast(device_type='cuda', dtype=amp_dtype, enabled=CONFIG['amp']):
                preds = model(images)
                feat_H = preds['landmark_heatmaps'].shape[2]
                _, comp = total_loss(
                    preds['landmark_heatmaps'], preds['landmark_coords'],
                    preds['gaze_vector'], gt_lm, gt_gaze,
                    feat_H, feat_H,
                    lam_lm=cfg['lam_lm'], lam_gaze=cfg['lam_gaze'],
                    sigma=cfg['sigma'])

            pred_px = preds['landmark_coords'] * (224 / feat_H)
            px_err = torch.mean(torch.norm(pred_px - gt_lm_px, dim=-1))

            val_metrics['total'] += comp['total_loss'].item()
            val_metrics['lm'] += comp['landmark_loss'].item()
            val_metrics['ang_deg'] += comp['angular_loss_deg'].item()
            val_metrics['lm_px'] += px_err.item()
            n_val += 1

    for k in val_metrics:
        val_metrics[k] /= max(n_val, 1)

    scheduler.step()
    lr = optimizer.param_groups[0]['lr']

    # --- Log ---
    mem_gb = torch.cuda.max_memory_allocated() / 1e9
    mv_str = f" reproj={train_metrics['reproj']:.4f} mask={train_metrics['mask']:.4f}" if use_mv else ""

    print(f"Ep {epoch:3d} | P{phase} | lr {lr:.1e} | "
          f"T: loss={train_metrics['total']:.4f} ang={train_metrics['ang_deg']:.1f}deg{mv_str} | "
          f"V: loss={val_metrics['total']:.4f} px={val_metrics['lm_px']:.1f} "
          f"ang={val_metrics['ang_deg']:.1f}deg | "
          f"mem={mem_gb:.1f}GB")

    history['train_total'].append(train_metrics['total'])
    history['val_total'].append(val_metrics['total'])
    history['train_ang'].append(train_metrics['ang_deg'])
    history['val_ang'].append(val_metrics['ang_deg'])

    csv_writer.writerow([
        epoch, phase, f"{lr:.2e}",
        f"{train_metrics['total']:.6f}", f"{train_metrics['lm']:.6f}",
        f"{train_metrics['ang_deg']:.4f}",
        f"{train_metrics['reproj']:.6f}", f"{train_metrics['mask']:.6f}",
        f"{val_metrics['total']:.6f}", f"{val_metrics['lm']:.6f}",
        f"{val_metrics['ang_deg']:.4f}", f"{val_metrics['lm_px']:.4f}",
    ])
    csv_file.flush()

    # --- Save best ---
    if val_metrics['total'] < best_val_loss:
        best_val_loss = val_metrics['total']
        state_dict = (model._orig_mod.state_dict()
                      if hasattr(model, '_orig_mod') else model.state_dict())
        torch.save({
            'epoch': epoch, 'phase': phase,
            'model_state_dict': state_dict,
            'val_loss': best_val_loss,
            'val_lm_px': val_metrics['lm_px'],
            'val_ang_deg': val_metrics['ang_deg'],
        }, os.path.join(output_dir, 'best_model.pt'))
        print(f"  -> Saved best (val_loss={best_val_loss:.4f})")

    # --- Periodic checkpoint ---
    if epoch % 5 == 0:
        state_dict = (model._orig_mod.state_dict()
                      if hasattr(model, '_orig_mod') else model.state_dict())
        torch.save({
            'epoch': epoch, 'phase': phase,
            'model_state_dict': state_dict,
            'optimizer_state_dict': optimizer.state_dict(),
            'scaler_state_dict': scaler.state_dict(),
        }, os.path.join(output_dir, f'ckpt_ep{epoch}.pt'))

csv_file.close()
print(f"\nTraining complete. Best val loss: {best_val_loss:.4f}")
print(f"Results: {output_dir}")

## 7. Monitoring & Visualization

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curves
axes[0].plot(history['train_total'], label='Train')
axes[0].plot(history['val_total'], label='Val')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Total Loss')
axes[0].set_title('Training Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Angular error
axes[1].plot(history['train_ang'], label='Train')
axes[1].plot(history['val_ang'], label='Val')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Angular Error (deg)')
axes[1].set_title('Gaze Angular Error')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Mark phase boundaries
for ax in axes:
    ax.axvline(x=5, color='gray', linestyle='--', alpha=0.5, label='Phase 2')
    ax.axvline(x=15, color='gray', linestyle=':', alpha=0.5, label='Phase 3')

plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'training_curves.png'), dpi=150)
plt.show()

## 8. Save & Export

In [ ]:
# Option A: Copy to Google Drive
from google.colab import drive
drive.mount('/content/drive')

drive_output = '/content/drive/MyDrive/RayNet_results'
!cp -r {output_dir} {drive_output}/
print(f"Results saved to Google Drive: {drive_output}")

In [ ]:
# Option B: Push model to Hugging Face Hub
from huggingface_hub import HfApi

api = HfApi()
model_repo = 'YOUR_USERNAME/raynet-v2'  # <-- EDIT THIS

api.create_repo(model_repo, repo_type='model', private=True, exist_ok=True)
api.upload_file(
    path_or_fileobj=os.path.join(output_dir, 'best_model.pt'),
    path_in_repo='best_model.pt',
    repo_id=model_repo,
    repo_type='model',
)
print(f"Model pushed to: https://huggingface.co/{model_repo}")

---

## Appendix: Create WebDataset Shards

Run this section **once** to convert your local dataset to shards and push to HF Hub. After that, use streaming mode above.

In [ ]:
# ====================================================================
# Create shards from local dataset (run once)
# ====================================================================

from RayNet.dataset import GazeGeneDataset
from RayNet.webdataset_utils import create_webdataset_shards, push_shards_to_hub

LOCAL_DATA_DIR = '/path/to/gazegene'  # <-- EDIT THIS
SHARD_DIR = '/content/shards'
HF_REPO = 'YOUR_USERNAME/gazegene-webdataset'  # <-- EDIT THIS

# --- Train split ---
train_ds = GazeGeneDataset(
    base_dir=LOCAL_DATA_DIR,
    subject_ids=list(range(1, 47)),
    eye='L', augment=False)

create_webdataset_shards(
    train_ds, os.path.join(SHARD_DIR, 'train'),
    samples_per_shard=1000, split='train', multiview_grouped=True)

# --- Val split ---
val_ds = GazeGeneDataset(
    base_dir=LOCAL_DATA_DIR,
    subject_ids=list(range(47, 57)),
    eye='L', augment=False)

create_webdataset_shards(
    val_ds, os.path.join(SHARD_DIR, 'val'),
    samples_per_shard=1000, split='val', multiview_grouped=True)

# --- Push to HF Hub ---
push_shards_to_hub(os.path.join(SHARD_DIR, 'train'), HF_REPO, split='train')
push_shards_to_hub(os.path.join(SHARD_DIR, 'val'), HF_REPO, split='val')

print("Shards created and pushed!")